In [1]:
from pathlib import Path

import pandas as pd

# Gather every yearly CSV whose filename ends with the year
data_dir = Path(".")
csv_files = sorted(
    path for path in data_dir.glob("world_happiness_*.csv")
    if path.stem.split("_")[-1].isdigit()
)

if not csv_files:
    raise FileNotFoundError("No yearly CSV files found in the current folder.")

frames = []
for csv_path in csv_files:
    year = int(csv_path.stem.split("_")[-1])
    df = pd.read_csv(csv_path, sep=";", decimal=",")
    # Later files call the score "Ladder score"; normalise the column name
    if "Ladder score" in df.columns and "Happiness score" not in df.columns:
        df = df.rename(columns={"Ladder score": "Happiness score"})
    df["Year"] = year
    frames.append(df)

combined_df = pd.concat(frames, ignore_index=True)

# Keep the key columns first; any extras follow at the end
column_order = [
    "Year",
    "Ranking",
    "Country",
    "Regional indicator",
    "Happiness score",
    "GDP per capita",
    "Social support",
    "Healthy life expectancy",
    "Freedom to make life choices",
    "Generosity",
    "Perceptions of corruption",
]
ordered_cols = [c for c in column_order if c in combined_df.columns]
remaining_cols = [c for c in combined_df.columns if c not in ordered_cols]
combined_df = combined_df[ordered_cols + remaining_cols]

combined_df.to_csv("world_happiness_combined.csv", index=False)
print("Combined rows:", len(combined_df))
print("Saved to world_happiness_combined.csv")

Combined rows: 1502
Saved to world_happiness_combined.csv
